# SAMPIC External-Trigger Diagnostics

Validate `TGxx` trigger provenance against `ATxx` timing and optional `ADxx` waveforms. This notebook is designed for sparse external-trigger runs: a trigger remains a tree entry even when it has no assigned hit.


In [ ]:
import os
import ROOT
import numpy as np
import matplotlib.pyplot as plt

DATA_FILE = os.getenv('SAMPIC_OUTPUT', '../../output.root')
LIB_DIR = '../../build/lib'
for name in ('libanalysis_pipeline_core.so',
             'libunpacker_data_products_core.so',
             'libunpacker_data_products_sampic.so'):
    status = ROOT.gSystem.Load(os.path.join(LIB_DIR, name))
    if status < 0:
        raise RuntimeError(f'Could not load {name}')

root_file = ROOT.TFile.Open(DATA_FILE)
if not root_file or root_file.IsZombie():
    raise RuntimeError(f'Could not open {DATA_FILE}')
tree = root_file.Get('events')
required = {'sampic_trigger_metadata', 'has_sampic_trigger_metadata',
            'sampic_event_timing', 'has_sampic_event_timing'}
branches = {branch.GetName() for branch in tree.GetListOfBranches()}
missing = required - branches
if missing:
    raise RuntimeError(f'Missing required branches: {sorted(missing)}')
print(f'Loaded {tree.GetEntries()} physics entries from {DATA_FILE}')


In [ ]:
records = []
for entry_idx in range(tree.GetEntries()):
    tree.GetEntry(entry_idx)
    if not bool(tree.has_sampic_trigger_metadata):
        continue
    tg = tree.sampic_trigger_metadata
    has_timing = bool(tree.has_sampic_event_timing)
    has_event = bool(getattr(tree, 'has_sampic_event', False))
    records.append({
        'entry': entry_idx,
        'fpga_id': int(tg.fpga_trigger_id),
        'external_id': int(tg.external_trigger_id),
        'trigger_index': int(tg.trigger_index_in_sampic_event),
        'assigned_hits': int(tg.assigned_hits),
        'ambiguous_hits': int(tg.ambiguous_hits),
        'trigger_ns': float(tg.trigger_timestamp_ns),
        'hit_reference_ns': float(tg.hit_reference_timestamp_ns),
        'timing_nhits': int(tree.sampic_event_timing.nhits) if has_timing else -1,
        'has_waveform': has_event,
    })

if not records:
    raise RuntimeError('No TG metadata records were decoded')

def array(name, dtype=float):
    return np.asarray([record[name] for record in records], dtype=dtype)

fpga_ids = array('fpga_id', int)
external_ids = array('external_id', int)
trigger_indices = array('trigger_index', int)
assigned_hits = array('assigned_hits', int)
ambiguous_hits = array('ambiguous_hits', int)
trigger_ns = array('trigger_ns')
hit_reference_ns = array('hit_reference_ns')
timing_nhits = array('timing_nhits', int)
has_waveform = array('has_waveform', bool)


In [ ]:
id_steps_mod_256 = np.diff(fpga_ids) % 256
intervals_ns = np.diff(trigger_ns)
positive_intervals_ns = intervals_ns[intervals_ns > 0]

print(f'TG records: {len(records)}')
print(f'FPGA ID range: {fpga_ids.min()}..{fpga_ids.max()} (8-bit rollover aware)')
print(f'Continuous FPGA IDs modulo 256: {np.all(id_steps_mod_256 == 1)}')
print(f'External trigger IDs observed: {np.unique(external_ids).tolist()}')
print(f'Waveform-bearing entries: {has_waveform.sum()}')
print(f'Assigned hits: {assigned_hits.sum()}')
print(f'Ambiguous hits: {ambiguous_hits.sum()}')
print(f'TG assigned_hits equals AT nhits: {np.array_equal(assigned_hits, timing_nhits)}')
if positive_intervals_ns.size:
    print(f'Mean trigger interval: {positive_intervals_ns.mean() * 1e-6:.6f} ms')
    print(f'Mean trigger rate: {1e9 / positive_intervals_ns.mean():.6f} Hz')


## Trigger continuity and cadence

The FPGA trigger ID is an 8-bit counter, so continuity is checked modulo 256. The trigger-index field identifies a trigger within its parent SAMPIC acquisition record and may reset independently.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].step(np.arange(fpga_ids.size), fpga_ids, where='mid')
axes[0, 0].set_title('FPGA trigger ID (visible rollover)')
axes[0, 0].set_xlabel('Tree entry')
axes[0, 0].set_ylabel('ID')
axes[0, 1].plot(trigger_indices, '.', markersize=3)
axes[0, 1].set_title('Trigger index within SAMPIC event')
axes[0, 1].set_xlabel('Tree entry')
axes[0, 1].set_ylabel('Index')
axes[1, 0].plot(intervals_ns * 1e-6, '.', markersize=3)
axes[1, 0].set_title('Inter-trigger interval')
axes[1, 0].set_xlabel('Trigger pair')
axes[1, 0].set_ylabel('Interval [ms]')
axes[1, 1].hist(positive_intervals_ns * 1e-6, bins=50)
axes[1, 1].set_title('Trigger interval distribution')
axes[1, 1].set_xlabel('Interval [ms]')
axes[1, 1].set_ylabel('Count')
plt.tight_layout()
plt.show()


## Hit assignment and timestamp residuals

`assigned_hits` should agree with `ATxx.nhits`. The hit-reference residual is shown only for triggers that received hits; zero-hit records intentionally carry no waveform product.


In [ ]:
hit_mask = assigned_hits > 0
residual_ns = hit_reference_ns[hit_mask] - trigger_ns[hit_mask]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(assigned_hits, bins=np.arange(assigned_hits.max() + 2) - 0.5)
axes[0].set_title('Assigned hits per trigger')
axes[0].set_xlabel('Hits')
axes[0].set_ylabel('Triggers')
axes[1].plot(np.flatnonzero(hit_mask), assigned_hits[hit_mask], 'o')
axes[1].set_title('Triggers receiving hits')
axes[1].set_xlabel('Tree entry')
axes[1].set_ylabel('Assigned hits')
if residual_ns.size:
    axes[2].plot(np.flatnonzero(hit_mask), residual_ns, 'o')
axes[2].set_title('Hit reference − trigger timestamp')
axes[2].set_xlabel('Tree entry')
axes[2].set_ylabel('Residual [ns]')
plt.tight_layout()
plt.show()

print('Hit-bearing entries:')
for index in np.flatnonzero(hit_mask):
    print(f"  entry={index:4d} fpga_id={fpga_ids[index]:3d} "
          f"trigger_index={trigger_indices[index]:3d} hits={assigned_hits[index]:2d} "
          f"residual={hit_reference_ns[index] - trigger_ns[index]:.3f} ns")
